In [ ]:
import pandas as pd

# Load the raw mandi master dataset before cleaning it
# This gives us the source data to inspect, validate, and standardize.
df = pd.read_csv("track3_mandi_master.csv")

# Preview the first few rows to understand the raw structure and column names
# before applying normalization rules.
df.head()

,mandi_id,mandi_name,district,state,mandi_type,total_area_acres
0,MANDI001,Hyderabad Mandi,ludhiana,Punjab,Private,11.0
1,MANDI006,Kochi Mandi,Amritsar,Punjab,Direct,34.0
2,MANDI037,Jorhat Grain Market,Kurukshetra,Haryana,APMC,NaN
3,MANDI046,Baranagar Market,NaN,Uttar Pradesh,Private,13.0
4,MANDI014,Bathinda Grain Market,Patiala,Punjab,PRIVATE,25.0


In [ ]:
# Check dataset size and schema before cleaning
# This helps identify missing values, null counts, and data types.
print("Shape:", df.shape)

print("\nColumn information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Shape: (60, 6)

Column information:
<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   mandi_id          60 non-null     str    
 1   mandi_name        60 non-null     str    
 2   district          56 non-null     str    
 3   state             56 non-null     str    
 4   mandi_type        49 non-null     str    
 5   total_area_acres  54 non-null     float64
dtypes: float64(1), str(5)
memory usage: 2.9 KB

Missing values:
mandi_id             0
mandi_name           0
district             4
state                4
mandi_type          11
total_area_acres     6
dtype: int64

Duplicate rows: 3


In [ ]:
# Show duplicate rows to confirm whether repeated records need to be removed
# Duplicate entries can distort counts and analysis if left in the dataset.
df[df.duplicated()]

,mandi_id,mandi_name,district,state,mandi_type,total_area_acres
9,MANDI001,Hyderabad Mandi,ludhiana,Punjab,Private,11.0
23,MANDI006,Kochi Mandi,Amritsar,Punjab,Direct,34.0
52,MANDI031,Nangloi Jat Mandi,Sirsa,Haryana,APMC,22.0


In [ ]:
# Remove exact duplicate rows so each mandi appears only once in the cleaned dataset
# This is a common first cleaning step in master datasets.
df = df.drop_duplicates()

print("Duplicate rows after cleaning:", df.duplicated().sum())
print("New shape:", df.shape)

Duplicate rows after cleaning: 0
New shape: (57, 6)


In [ ]:
# Inspect rows with any missing field to identify where cleaning is needed
# This helps decide which columns require imputation or value standardization.
df[df.isnull().any(axis=1)]

,mandi_id,mandi_name,district,state,mandi_type,total_area_acres
2,MANDI037,Jorhat Grain Market,Kurukshetra,Haryana,APMC,NaN
3,MANDI046,Baranagar Market,NaN,Uttar Pradesh,Private,13.0
6,MANDI034,Ludhiana Grain Market,Ambala,Haryana,NaN,11.0
10,MANDI047,Orai Market,Muzaffarnagar,Uttar Pradesh,APMC,NaN
12,MANDI032,Chapra Grain Market,NaN,Haryana,PRIVATE,20.0
14,MANDI053,Panihati Market,Saharanpur,Uttar Pradesh,APMC,NaN
15,MANDI018,Jaunpur Mandi,MOGA,Punjab,NaN,9.0
25,MANDI057,Gorakhpur APMC,Bareilly,NaN,Direct,7.0
26,MANDI016,Kulti APMC,Bathinda,Punjab,NaN,42.0
27,MANDI028,Machilipatnam APMC,Hisar,Haryana,NaN,33.0


In [ ]:
# Standardize text fields to improve consistency across spelling and casing variations
# Strip leading/trailing spaces and convert to title case for cleaner labels.
text_columns = ["mandi_name", "district", "state", "mandi_type"]

for col in text_columns:
    df[col] = df[col].str.strip().str.title()

df[text_columns].head()

,mandi_name,district,state,mandi_type
0,Hyderabad Mandi,Ludhiana,Punjab,Private
1,Kochi Mandi,Amritsar,Punjab,Direct
2,Jorhat Grain Market,Kurukshetra,Haryana,Apmc
3,Baranagar Market,NaN,Uttar Pradesh,Private
4,Bathinda Grain Market,Patiala,Punjab,Private


In [ ]:
# Review the standardized categorical values to confirm the text cleaning worked as intended.
print("District values:")
print(df["district"].unique())

print("\nState values:")
print(df["state"].unique())

print("\nMandi type values:")
print(df["mandi_type"].unique())

print("\nTotal area summary:")
print(df["total_area_acres"].describe())

District values:
<StringArray>
[     'Ludhiana',      'Amritsar',   'Kurukshetra',             nan,
       'Patiala',      'Bareilly',        'Ambala', 'Muzaffarnagar',
    'Saharanpur',          'Moga',     'Jalandhar',     'Fatehabad',
          'Agra',     'Ferozepur',         'Hisar',      'Bathinda',
         'Sirsa',        'Karnal',        'Meerut']
Length: 19, dtype: str

State values:
<StringArray>
['Punjab', 'Haryana', 'Uttar Pradesh', nan]
Length: 4, dtype: str

Mandi type values:
<StringArray>
['Private', 'Direct', 'Apmc', nan]
Length: 4, dtype: str

Total area summary:
count    51.000000
mean     27.921569
std      12.871431
min       5.000000
25%      18.000000
50%      28.000000
75%      39.000000
max      48.000000
Name: total_area_acres, dtype: float64


In [ ]:
# Normalize mandi type values to uppercase for consistent categorical values
# This reduces issues caused by mixed casing such as 'Apmc' and 'APMC'.
df["mandi_type"] = df["mandi_type"].str.upper()

print(df["mandi_type"].value_counts(dropna=False))

mandi_type
APMC       20
PRIVATE    17
NaN        11
DIRECT      9
Name: count, dtype: int64


In [ ]:
# Fill missing mandi types with a default value so the column remains usable for analysis
# This prevents null categories from breaking downstream aggregation and filtering.
df["mandi_type"] = df["mandi_type"].fillna("APMC")

print(df["mandi_type"].value_counts())

mandi_type
APMC       31
PRIVATE    17
DIRECT      9
Name: count, dtype: int64


In [ ]:
# Check rows where district is missing so we can decide how to impute them
# Missing location values can be filled from state-level patterns or set to a default label.
df[df["district"].isnull()][["mandi_id", "mandi_name", "district", "state"]]

,mandi_id,mandi_name,district,state
3,MANDI046,Baranagar Market,NaN,Uttar Pradesh
12,MANDI032,Chapra Grain Market,NaN,Haryana
31,MANDI017,Parbhani Market,NaN,Punjab
59,MANDI039,Anand Grain Market,NaN,Haryana


In [ ]:
# Inspect district frequency to understand the observed categories before filling missing values.
df["district"].value_counts()

district
Ludhiana         4
Amritsar         4
Patiala          4
Muzaffarnagar    4
Saharanpur       4
Karnal           4
Kurukshetra      3
Bareilly         3
Ambala           3
Fatehabad        3
Hisar            3
Sirsa            3
Moga             2
Jalandhar        2
Agra             2
Ferozepur        2
Bathinda         2
Meerut           1
Name: count, dtype: int64

In [ ]:
# Fill missing district values using the most common district within each state
# This preserves geographic consistency without creating random or unrealistic labels.
df["district"] = df.groupby("state")["district"].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "Unknown")
)

print("Missing districts after cleaning:")
print(df["district"].isnull().sum())

Missing districts after cleaning:
4


In [ ]:
# Verify whether any district values still remain missing after the state-based imputation.
df[df["district"].isnull()][["mandi_id", "mandi_name", "state"]]

,mandi_id,mandi_name,state
25,MANDI057,Gorakhpur Apmc,NaN
36,MANDI054,Nangloi Jat Grain Market,NaN
39,MANDI030,Patiala Apmc,NaN
41,MANDI002,Solapur Mandi,NaN


In [ ]:
# Final fallback for remaining missing location values
# These columns are essential for filtering and grouping, so unknown values are set explicitly.
df["district"] = df["district"].fillna("Unknown")
df["state"] = df["state"].fillna("Unknown")

print("Missing districts:", df["district"].isnull().sum())
print("Missing states:", df["state"].isnull().sum())

Missing districts: 0
Missing states: 0


In [ ]:
# Review the final state distribution after imputing missing values
# This confirms the cleaned dataset still has realistic geographic coverage.
print(df["state"].value_counts())

state
Punjab           20
Haryana          20
Uttar Pradesh    13
Unknown           4
Name: count, dtype: int64


In [ ]:
# Check the numeric area column before filling missing numbers
# This tells us whether the area variable has nulls and how spread out the values are.
print("Missing total area:", df["total_area_acres"].isnull().sum())
print("\nArea statistics:")
print(df["total_area_acres"].describe())

Missing total area: 6

Area statistics:
count    51.000000
mean     27.921569
std      12.871431
min       5.000000
25%      18.000000
50%      28.000000
75%      39.000000
max      48.000000
Name: total_area_acres, dtype: float64


In [ ]:
# Fill missing area values with the median to avoid skewing the distribution with extreme values
# The median is more robust than the mean for agricultural land area data.
median_area = df["total_area_acres"].median()

df["total_area_acres"] = df["total_area_acres"].fillna(median_area)

print("Median area used:", median_area)
print("Missing total area after cleaning:", df["total_area_acres"].isnull().sum())

Median area used: 28.0
Missing total area after cleaning: 0


In [ ]:
# Final validation check after all cleaning steps
# This confirms the dataset has no remaining nulls or duplicate records before export.
print("Final Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

Final Shape: (57, 6)

Missing values:
mandi_id            0
mandi_name          0
district            0
state               0
mandi_type          0
total_area_acres    0
dtype: int64

Duplicate rows:
0

Data types:
mandi_id                str
mandi_name              str
district                str
state                   str
mandi_type              str
total_area_acres    float64
dtype: object


In [ ]:
# Display the cleaned dataset for a final visual review
# This is the last manual check before naming and exporting the final columns.
df

,mandi_id,mandi_name,district,state,mandi_type,total_area_acres
0,MANDI001,Hyderabad Mandi,Ludhiana,Punjab,PRIVATE,11.0
1,MANDI006,Kochi Mandi,Amritsar,Punjab,DIRECT,34.0
2,MANDI037,Jorhat Grain Market,Kurukshetra,Haryana,APMC,28.0
3,MANDI046,Baranagar Market,Muzaffarnagar,Uttar Pradesh,PRIVATE,13.0
4,MANDI014,Bathinda Grain Market,Patiala,Punjab,PRIVATE,25.0
5,MANDI055,Chandigarh Apmc,Bareilly,Uttar Pradesh,APMC,12.0
6,MANDI034,Ludhiana Grain Market,Ambala,Haryana,APMC,11.0
7,MANDI049,Machilipatnam Mandi,Muzaffarnagar,Uttar Pradesh,PRIVATE,39.0
8,MANDI013,Aurangabad Market,Patiala,Punjab,PRIVATE,25.0
10,MANDI047,Orai Market,Muzaffarnagar,Uttar Pradesh,APMC,28.0


In [ ]:
# Reset the index to start from 1 so the table reads cleanly
# Then convert column names to a standardized format for downstream use.
df.index = range(1, len(df) + 1)

df.columns = [
    "Mandi_ID",
    "Mandi_Name",
    "District",
    "State",
    "Mandi_Type",
    "Total_Area_Acres"
]

df

,Mandi_ID,Mandi_Name,District,State,Mandi_Type,Total_Area_Acres
1,MANDI001,Hyderabad Mandi,Ludhiana,Punjab,PRIVATE,11.0
2,MANDI006,Kochi Mandi,Amritsar,Punjab,DIRECT,34.0
3,MANDI037,Jorhat Grain Market,Kurukshetra,Haryana,APMC,28.0
4,MANDI046,Baranagar Market,Muzaffarnagar,Uttar Pradesh,PRIVATE,13.0
5,MANDI014,Bathinda Grain Market,Patiala,Punjab,PRIVATE,25.0
6,MANDI055,Chandigarh Apmc,Bareilly,Uttar Pradesh,APMC,12.0
7,MANDI034,Ludhiana Grain Market,Ambala,Haryana,APMC,11.0
8,MANDI049,Machilipatnam Mandi,Muzaffarnagar,Uttar Pradesh,PRIVATE,39.0
9,MANDI013,Aurangabad Market,Patiala,Punjab,PRIVATE,25.0
10,MANDI047,Orai Market,Muzaffarnagar,Uttar Pradesh,APMC,28.0


In [ ]:
# Add a serial number column to keep a readable row index in the cleaned dataset
# This is useful for referencing records in reports and manual validation.
df.insert(0, "Serial_No", range(1, len(df) + 1))

# Rename the remaining columns
# The final output should use consistent, business-friendly names.
df.columns = [
    "Serial_No",
    "Mandi_ID",
    "Mandi_Name",
    "District",
    "State",
    "Mandi_Type",
    "Total_Area_Acres"
]

# Display the final dataset
# The dataset is now ready for review and export.
df

,Serial_No,Mandi_ID,Mandi_Name,District,State,Mandi_Type,Total_Area_Acres
1,1,MANDI001,Hyderabad Mandi,Ludhiana,Punjab,PRIVATE,11.0
2,2,MANDI006,Kochi Mandi,Amritsar,Punjab,DIRECT,34.0
3,3,MANDI037,Jorhat Grain Market,Kurukshetra,Haryana,APMC,28.0
4,4,MANDI046,Baranagar Market,Muzaffarnagar,Uttar Pradesh,PRIVATE,13.0
5,5,MANDI014,Bathinda Grain Market,Patiala,Punjab,PRIVATE,25.0
6,6,MANDI055,Chandigarh Apmc,Bareilly,Uttar Pradesh,APMC,12.0
7,7,MANDI034,Ludhiana Grain Market,Ambala,Haryana,APMC,11.0
8,8,MANDI049,Machilipatnam Mandi,Muzaffarnagar,Uttar Pradesh,PRIVATE,39.0
9,9,MANDI013,Aurangabad Market,Patiala,Punjab,PRIVATE,25.0
10,10,MANDI047,Orai Market,Muzaffarnagar,Uttar Pradesh,APMC,28.0


In [ ]:
# Remove the temporary serial column if it was already created earlier
# This keeps only the final, clean column structure.
if "Serial_No" in df.columns:
    df = df.drop(columns=["Serial_No"])

# Reset the DataFrame index after dropping columns
# This ensures row numbering stays consistent with the cleaned output.
df.index = range(1, len(df) + 1)

# Rename columns
# This final naming step prepares the dataset for consistent downstream use.
df.columns = [
    "Mandi_ID",
    "Mandi_Name",
    "District",
    "State",
    "Mandi_Type",
    "Total_Area_Acres"
]

df

,Mandi_ID,Mandi_Name,District,State,Mandi_Type,Total_Area_Acres
1,MANDI001,Hyderabad Mandi,Ludhiana,Punjab,PRIVATE,11.0
2,MANDI006,Kochi Mandi,Amritsar,Punjab,DIRECT,34.0
3,MANDI037,Jorhat Grain Market,Kurukshetra,Haryana,APMC,28.0
4,MANDI046,Baranagar Market,Muzaffarnagar,Uttar Pradesh,PRIVATE,13.0
5,MANDI014,Bathinda Grain Market,Patiala,Punjab,PRIVATE,25.0
6,MANDI055,Chandigarh Apmc,Bareilly,Uttar Pradesh,APMC,12.0
7,MANDI034,Ludhiana Grain Market,Ambala,Haryana,APMC,11.0
8,MANDI049,Machilipatnam Mandi,Muzaffarnagar,Uttar Pradesh,PRIVATE,39.0
9,MANDI013,Aurangabad Market,Patiala,Punjab,PRIVATE,25.0
10,MANDI047,Orai Market,Muzaffarnagar,Uttar Pradesh,APMC,28.0


In [ ]:
# Remove the temporary serial column if it exists from earlier exploratory steps
# This keeps the final dataset clean and free from intermediate columns.
if "Serial_No" in df.columns:
    df = df.drop(columns=["Serial_No"])

# Create fresh sequential Mandi IDs so each record has a consistent unique identifier
# This standardizes the key column for joining with other datasets.
df["Mandi_ID"] = [
    f"MANDI{i:03d}" for i in range(1, len(df) + 1)
]

# Set the final column order for a clean export-ready dataset
# This improves consistency when sharing or merging with other files.
df = df[
    ["Mandi_ID", "Mandi_Name", "District",
     "State", "Mandi_Type", "Total_Area_Acres"]
]

df

,Mandi_ID,Mandi_Name,District,State,Mandi_Type,Total_Area_Acres
1,MANDI001,Hyderabad Mandi,Ludhiana,Punjab,PRIVATE,11.0
2,MANDI002,Kochi Mandi,Amritsar,Punjab,DIRECT,34.0
3,MANDI003,Jorhat Grain Market,Kurukshetra,Haryana,APMC,28.0
4,MANDI004,Baranagar Market,Muzaffarnagar,Uttar Pradesh,PRIVATE,13.0
5,MANDI005,Bathinda Grain Market,Patiala,Punjab,PRIVATE,25.0
6,MANDI006,Chandigarh Apmc,Bareilly,Uttar Pradesh,APMC,12.0
7,MANDI007,Ludhiana Grain Market,Ambala,Haryana,APMC,11.0
8,MANDI008,Machilipatnam Mandi,Muzaffarnagar,Uttar Pradesh,PRIVATE,39.0
9,MANDI009,Aurangabad Market,Patiala,Punjab,PRIVATE,25.0
10,MANDI010,Orai Market,Muzaffarnagar,Uttar Pradesh,APMC,28.0


In [ ]:
# Save a copy of the data BEFORE changing Mandi_ID
# This protects the original text fields while we verify the renumbering logic.
data_before = df[
    ["Mandi_Name", "District", "State", "Mandi_Type", "Total_Area_Acres"]
].copy()

# Remove Serial_No if it exists
# This is a cleanup step before finalizing the ID scheme.
if "Serial_No" in df.columns:
    df = df.drop(columns=["Serial_No"])

# Renumber Mandi_ID according to the CURRENT ROW ORDER
# This ensures IDs remain sequential and consistent after all cleaning steps.
df["Mandi_ID"] = [
    f"MANDI{i:03d}" for i in range(1, len(df) + 1)
]

# Set exact column names and order
# This creates the final master-table structure used for analysis.
df.columns = [
    "Mandi_ID",
    "Mandi_Name",
    "District",
    "State",
    "Mandi_Type",
    "Total_Area_Acres"
]

# Verify that all other data stayed exactly the same
# This safeguards against accidental changes during ID renumbering.
data_after = df[
    ["Mandi_Name", "District", "State", "Mandi_Type", "Total_Area_Acres"]
].copy()

if data_before.reset_index(drop=True).equals(data_after.reset_index(drop=True)):
    print("✅ All other data is unchanged!")
else:
    print("❌ Something changed — DO NOT SAVE yet.")

# Verify IDs
print("\nFirst 10 IDs:")
print(df["Mandi_ID"].head(10).tolist())

print("\nLast 10 IDs:")
print(df["Mandi_ID"].tail(10).tolist())

print("\nFinal shape:", df.shape)

df

✅ All other data is unchanged!

First 10 IDs:
['MANDI001', 'MANDI002', 'MANDI003', 'MANDI004', 'MANDI005', 'MANDI006', 'MANDI007', 'MANDI008', 'MANDI009', 'MANDI010']

Last 10 IDs:
['MANDI048', 'MANDI049', 'MANDI050', 'MANDI051', 'MANDI052', 'MANDI053', 'MANDI054', 'MANDI055', 'MANDI056', 'MANDI057']

Final shape: (57, 6)


,Mandi_ID,Mandi_Name,District,State,Mandi_Type,Total_Area_Acres
1,MANDI001,Hyderabad Mandi,Ludhiana,Punjab,PRIVATE,11.0
2,MANDI002,Kochi Mandi,Amritsar,Punjab,DIRECT,34.0
3,MANDI003,Jorhat Grain Market,Kurukshetra,Haryana,APMC,28.0
4,MANDI004,Baranagar Market,Muzaffarnagar,Uttar Pradesh,PRIVATE,13.0
5,MANDI005,Bathinda Grain Market,Patiala,Punjab,PRIVATE,25.0
6,MANDI006,Chandigarh Apmc,Bareilly,Uttar Pradesh,APMC,12.0
7,MANDI007,Ludhiana Grain Market,Ambala,Haryana,APMC,11.0
8,MANDI008,Machilipatnam Mandi,Muzaffarnagar,Uttar Pradesh,PRIVATE,39.0
9,MANDI009,Aurangabad Market,Patiala,Punjab,PRIVATE,25.0
10,MANDI010,Orai Market,Muzaffarnagar,Uttar Pradesh,APMC,28.0


In [ ]:
# Reload the original dataset to begin a fresh cleaning workflow
# This ensures the final cleaned version is based on the raw source data.
import pandas as pd

df = pd.read_csv("track3_mandi_master.csv")

print("Original shape:", df.shape)
df.head()

Original shape: (60, 6)


,mandi_id,mandi_name,district,state,mandi_type,total_area_acres
0,MANDI001,Hyderabad Mandi,ludhiana,Punjab,Private,11.0
1,MANDI006,Kochi Mandi,Amritsar,Punjab,Direct,34.0
2,MANDI037,Jorhat Grain Market,Kurukshetra,Haryana,APMC,NaN
3,MANDI046,Baranagar Market,NaN,Uttar Pradesh,Private,13.0
4,MANDI014,Bathinda Grain Market,Patiala,Punjab,PRIVATE,25.0


In [ ]:
# Remove duplicate rows before any other standardization
# Duplicate mandi records can otherwise lead to incorrect counts and joins.
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)
print("Duplicates remaining:", df.duplicated().sum())

Shape after removing duplicates: (57, 6)
Duplicates remaining: 0


In [ ]:
# Standardize mandi type values to uppercase for consistent reporting
# This reduces variation such as 'Apmc', 'apmc', and 'APMC'.
df["mandi_type"] = df["mandi_type"].str.upper()

# Fill missing mandi type values with a consistent default label
# This prevents null categories from breaking group-by and summary operations.
df["mandi_type"] = df["mandi_type"].fillna("APMC")

# Fill missing district and state entries with a placeholder label
# This maintains row completeness for later joins and filtering.
df["district"] = df["district"].fillna("Unknown")
df["state"] = df["state"].fillna("Unknown")

In [30]:
median_area = df["total_area_acres"].median()

df["total_area_acres"] = df["total_area_acres"].fillna(median_area)

print("Median area used:", median_area)
print("Missing area:", df["total_area_acres"].isnull().sum())

Median area used: 28.0
Missing area: 0


In [31]:
df = df.sort_values("mandi_id").reset_index(drop=True)

df.columns = [
    "Mandi_ID",
    "Mandi_Name",
    "District",
    "State",
    "Mandi_Type",
    "Total_Area_Acres"
]

df

,Mandi_ID,Mandi_Name,District,State,Mandi_Type,Total_Area_Acres
0,MANDI001,Hyderabad Mandi,ludhiana,Punjab,PRIVATE,11.0
1,MANDI002,Solapur Mandi,Ludhiana,Unknown,APMC,37.0
2,MANDI003,Vijayawada Mandi,Ludhiana,Punjab,APMC,19.0
3,MANDI004,Khandwa Grain Market,Ludhiana,Punjab,APMC,15.0
4,MANDI005,Bhilwara Grain Market,Amritsar,Punjab,DIRECT,29.0
5,MANDI006,Kochi Mandi,Amritsar,Punjab,DIRECT,34.0
6,MANDI007,Nashik Mandi,Amritsar,Punjab,PRIVATE,45.0
7,MANDI008,Bijapur Market,Amritsar,Punjab,APMC,23.0
8,MANDI009,Jamshedpur APMC,Jalandhar,Punjab,APMC,28.0
9,MANDI010,Dhule APMC,Jalandhar,Punjab,DIRECT,48.0


In [32]:
print("Final Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nFirst 10 rows:")
display(df.head(10))

print("\nCheck MANDI007:")
display(df[df["Mandi_ID"] == "MANDI007"])

Final Shape: (57, 6)

Missing values:
Mandi_ID            0
Mandi_Name          0
District            0
State               0
Mandi_Type          0
Total_Area_Acres    0
dtype: int64

Duplicate rows:
0

First 10 rows:


,Mandi_ID,Mandi_Name,District,State,Mandi_Type,Total_Area_Acres
0,MANDI001,Hyderabad Mandi,ludhiana,Punjab,PRIVATE,11.0
1,MANDI002,Solapur Mandi,Ludhiana,Unknown,APMC,37.0
2,MANDI003,Vijayawada Mandi,Ludhiana,Punjab,APMC,19.0
3,MANDI004,Khandwa Grain Market,Ludhiana,Punjab,APMC,15.0
4,MANDI005,Bhilwara Grain Market,Amritsar,Punjab,DIRECT,29.0
5,MANDI006,Kochi Mandi,Amritsar,Punjab,DIRECT,34.0
6,MANDI007,Nashik Mandi,Amritsar,Punjab,PRIVATE,45.0
7,MANDI008,Bijapur Market,Amritsar,Punjab,APMC,23.0
8,MANDI009,Jamshedpur APMC,Jalandhar,Punjab,APMC,28.0
9,MANDI010,Dhule APMC,Jalandhar,Punjab,DIRECT,48.0



Check MANDI007:


,Mandi_ID,Mandi_Name,District,State,Mandi_Type,Total_Area_Acres
6,MANDI007,Nashik Mandi,Amritsar,Punjab,PRIVATE,45.0


In [33]:
df.to_csv("cleaned_mandi_master.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [34]:
# Standardize district names
df["District"] = df["District"].str.strip().str.title()

# Display all district values to verify
print(df["District"].unique())

<StringArray>
[     'Ludhiana',      'Amritsar',     'Jalandhar',       'Patiala',
      'Bathinda',       'Unknown',          'Moga',     'Ferozepur',
        'Karnal',         'Hisar',         'Sirsa',        'Ambala',
   'Kurukshetra',     'Fatehabad',          'Agra',        'Meerut',
 'Muzaffarnagar',    'Saharanpur',      'Bareilly']
Length: 19, dtype: str


In [35]:
df

,Mandi_ID,Mandi_Name,District,State,Mandi_Type,Total_Area_Acres
0,MANDI001,Hyderabad Mandi,Ludhiana,Punjab,PRIVATE,11.0
1,MANDI002,Solapur Mandi,Ludhiana,Unknown,APMC,37.0
2,MANDI003,Vijayawada Mandi,Ludhiana,Punjab,APMC,19.0
3,MANDI004,Khandwa Grain Market,Ludhiana,Punjab,APMC,15.0
4,MANDI005,Bhilwara Grain Market,Amritsar,Punjab,DIRECT,29.0
5,MANDI006,Kochi Mandi,Amritsar,Punjab,DIRECT,34.0
6,MANDI007,Nashik Mandi,Amritsar,Punjab,PRIVATE,45.0
7,MANDI008,Bijapur Market,Amritsar,Punjab,APMC,23.0
8,MANDI009,Jamshedpur APMC,Jalandhar,Punjab,APMC,28.0
9,MANDI010,Dhule APMC,Jalandhar,Punjab,DIRECT,48.0


In [36]:
df.to_csv("cleaned_mandi_master.csv", index=False)

print("Updated cleaned dataset saved successfully!")

Updated cleaned dataset saved successfully!
